# Puzzle Visualization
Сравнение `beam_search` (baseline) и `BidirectionalSearcher` (новый) на каждой головоломке.

In [ ]:
import time
import os
import torch

from viz_utils import beam_search, show_fifteen, show_lights_out, show_rotate_slide
from core.searcher import BidirectionalSearcher
from core.encoder import StateEncoder
from models.value_net import ValueNet

MODEL_PATH = 'model.pt'
SCRAMBLE_LEN = 42
SEED = 42
BIDIR_TIMEOUT = 10.0  # секунд на задачу

def load_model():
    if not os.path.exists(MODEL_PATH):
        print('model.pt не найден — используем поиск без модели')
        return None
    ckpt = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
    model = ValueNet()
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    print('model.pt загружен')
    return model

encoder = StateEncoder()
model = load_model()

## Игра 15

In [ ]:
from gym import Fifteen2DEnv

env = Fifteen2DEnv()
state, _ = env.scramble(length=SCRAMBLE_LEN, seed=SEED)

beam_sol = beam_search(env)
print(f'beam_search:  {len(beam_sol)} ходов')
show_fifteen(state, beam_sol)

In [ ]:
searcher = BidirectionalSearcher(env, encoder, model)
bidir_sol = searcher.solve(state, deadline=time.time() + BIDIR_TIMEOUT)

if bidir_sol is None:
    print('bidir: не решено за отведённое время')
else:
    print(f'bidir A*:     {len(bidir_sol)} ходов  (beam: {len(beam_sol)})')
    show_fifteen(state, bidir_sol)

## Lights Out

In [ ]:
from gym import LightsOutEnv

env = LightsOutEnv()
state, _ = env.scramble(length=SCRAMBLE_LEN, seed=SEED)

beam_sol = beam_search(env)
print(f'beam_search:  {len(beam_sol)} ходов')
show_lights_out(state, beam_sol)

In [ ]:
searcher = BidirectionalSearcher(env, encoder, model)
bidir_sol = searcher.solve(state, deadline=time.time() + BIDIR_TIMEOUT)

if bidir_sol is None:
    print('bidir: не решено за отведённое время')
else:
    print(f'bidir A*:     {len(bidir_sol)} ходов  (beam: {len(beam_sol)})')
    show_lights_out(state, bidir_sol)

## Цилиндр (Варикон)

In [ ]:
from gym import RotateSlideEnv

env = RotateSlideEnv()
state, _ = env.scramble(length=SCRAMBLE_LEN, seed=SEED)

beam_sol = beam_search(env)
print(f'beam_search:  {len(beam_sol)} ходов')
show_rotate_slide(state, beam_sol)

In [ ]:
searcher = BidirectionalSearcher(env, encoder, model)
bidir_sol = searcher.solve(state, deadline=time.time() + BIDIR_TIMEOUT)

if bidir_sol is None:
    print('bidir: не решено за отведённое время')
else:
    print(f'bidir A*:     {len(bidir_sol)} ходов  (beam: {len(beam_sol)})')
    show_rotate_slide(state, bidir_sol)

## Сводный бенчмарк
Прогоняем несколько scramble и сравниваем длины решений.

In [ ]:
from gym import Fifteen2DEnv, LightsOutEnv, RotateSlideEnv

PUZZLES = [
    ('game_15_2d',    Fifteen2DEnv,    40),
    ('toggle_lights', LightsOutEnv,    25),
    ('cylinder_game', RotateSlideEnv,  60),
]
N_SEEDS = 5

print(f'{'Puzzle':<16} {'Seed':<6} {'Baseline':<10} {'Beam':<8} {'Bidir':<8} {'Score'}')
print('-' * 60)

for name, EnvClass, scramble_len in PUZZLES:
    env = EnvClass()
    searcher = BidirectionalSearcher(env, encoder, model)
    for seed in range(N_SEEDS):
        state, actions = env.scramble(length=scramble_len, seed=seed)
        baseline = len(actions)

        beam_sol = beam_search(env)
        bidir_sol = searcher.solve(state, deadline=time.time() + 5.0)

        bidir_len = len(bidir_sol) if bidir_sol else '-'
        score = f'{baseline / len(bidir_sol):.2f}' if bidir_sol else '0'
        print(f'{name:<16} {seed:<6} {baseline:<10} {len(beam_sol):<8} {str(bidir_len):<8} {score}')